<a href="https://colab.research.google.com/github/Plabon-Basak/FlyRank_AI/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Plabon-Basak/FlyRank_AI/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
**One row = one content page for one client on one reporting date.**

For this assignment, I use the March 2026 partition as the development and verification window. I use March rather than the `_sample` table because `_sample` contains the final month, June 2026, which should remain a sealed outcome window.

The decision grain for my lane is a content page. The daily warehouse grain lets me measure the page's search-performance signals at a specific reporting date before creating a review-priority ranking.

The ranking output is intended as decision-support: it prioritizes pages for human content/SEO review rather than automatically changing pages.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import getpass
import duckdb

# Get the Hugging Face READ token from Colab Secret.
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/**/*.parquet'"
    f")"
)

MARCH_START = "2026-03-01"
APRIL_START = "2026-04-01"

print("Warehouse connection ready.")
print("Development window:", MARCH_START, "to", "2026-03-31")
print("Decision grain: client × content × report_date")

Paste your Hugging Face READ token (hf_...): ··········
Warehouse connection ready.
Development window: 2026-03-01 to 2026-03-31
Decision grain: client × content × report_date


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
### Feature fields

The five features I will use for the first feature frame are:

* `gsc_impressions` — historical search impressions available before the ranking decision.
* `gsc_clicks` — historical search clicks available before the ranking decision.
* `gsc_avg_position` — historical average search position available before the ranking decision.
* `ga4_sessions` — observed analytics sessions available before the ranking decision, when GA4 data is available.
* `gsc_ctr` — historical click-through rate derived from observed search performance.

### Label / proxy

For the deliberate leakage experiment, I use a label/proxy derived from the observed performance outcome. The label is not treated as a feature because it represents the outcome being evaluated.

### Context

* `client_hash_id` — identifies the client for grouping and analysis, but is not a predictive feature.
* `content_hash_id` — identifies the content page and is used to preserve the decision grain.
* `report_date` — identifies the observation date and is used for time-window filtering.

### Excluded

I deliberately exclude any label-derived field, future outcome field, or field calculated from the outcome window. These fields would expose information that would not be available at the moment a page is ranked and would cause target leakage.

I also exclude the pseudonymous client and content IDs from the feature set because they are identifiers rather than meaningful page-performance signals.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Field contract for the first feature frame

feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "gsc_ctr",
]

context_columns = [
    "client_hash_id",
    "content_hash_id",
    "report_date",
]

print("Features:")
for col in feature_columns:
    print(" -", col)

print("\nContext:")
for col in context_columns:
    print(" -", col)

print("\nExcluded:")
print(" - label-derived fields")
print(" - future outcome fields")
print(" - client/content identifiers as predictive features")

Features:
 - gsc_impressions
 - gsc_clicks
 - gsc_avg_position
 - ga4_sessions
 - gsc_ctr

Context:
 - client_hash_id
 - content_hash_id
 - report_date

Excluded:
 - label-derived fields
 - future outcome fields
 - client/content identifiers as predictive features


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*
### Query 1 — Grain verification

I expect each combination of client, content page, and reporting date to occur once. The query searches for duplicate combinations. An empty result means no duplicate rows were observed in the March 2026 slice.


In [5]:
grain_check = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        COUNT(*) AS row_count
    FROM {FACT_DAILY}
    WHERE report_date >= DATE '{MARCH_START}'
      AND report_date < DATE '{APRIL_START}'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

display(grain_check)

if grain_check.empty:
    print("Result: no duplicate client × content × report_date combinations observed.")
else:
    print("Result: duplicate combinations were observed and require investigation.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,row_count


Result: no duplicate client × content × report_date combinations observed.


### Query 2 — March 2026 row count and date span

This query measures the size of my March 2026 slice and verifies its observed date boundaries. I use the measured values rather than assuming the partition contains every day.


In [6]:
march_summary = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {FACT_DAILY}
    WHERE report_date >= DATE '{MARCH_START}'
      AND report_date < DATE '{APRIL_START}'
""").df()

display(march_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


### Query 3 — GA4 availability

Some early warehouse rows contain zero-filled GA4 values when GA4 data was not available. Therefore, I use the explicit `ga4_data_available IS TRUE` flag rather than interpreting zero values as genuine zero activity.

The query measures how many March rows have GA4 data available.


In [7]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS rows_in_march,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS rows_with_ga4_available
    FROM {FACT_DAILY}
    WHERE report_date >= DATE '{MARCH_START}'
      AND report_date < DATE '{APRIL_START}'
""").df()

display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_in_march,rows_with_ga4_available
0,9841378,413966


## Five-feature frame

I build the feature frame from the March 2026 slice. The features represent information that can be observed before making a review-priority decision.

| Feature            | Available when?                                                                                                    |
| ------------------ | ------------------------------------------------------------------------------------------------------------------ |
| `gsc_impressions`  | Available when the decision is made because it represents previously observed search impressions.                  |
| `gsc_clicks`       | Available when the decision is made because it represents previously observed search clicks.                       |
| `gsc_avg_position` | Available when the decision is made because it represents previously observed search position.                     |
| `ga4_sessions`     | Available when the decision is made when GA4 data is available because it represents previously observed sessions. |
| `gsc_ctr`          | Available when the decision is made because it summarizes previously observed clicks relative to impressions.      |

The frame retains `client_hash_id`, `content_hash_id`, and `report_date` only as context columns. They are not used as predictive features.


In [9]:
feature_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_sessions,
        gsc_clicks / NULLIF(gsc_impressions, 0) AS gsc_ctr
    FROM {FACT_DAILY}
    WHERE report_date >= DATE '{MARCH_START}'
      AND report_date < DATE '{APRIL_START}'
""").df()

print("Feature frame shape:", feature_frame.shape)

display(feature_frame.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (9841378, 8)


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,gsc_ctr
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000,<NA>,0.000
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0.000000,<NA>,0.000
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,4.928000,<NA>,0.008
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,4.000000,<NA>,0.000
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,2.272727,<NA>,0.000


## Deliberate leakage experiment

I intentionally add a label-derived feature to demonstrate target leakage. Because this feature contains the answer used to evaluate the model, a quick score can become artificially close to perfect.

This result is deliberately invalid. The leakage feature is then removed, and only the honest feature set is retained.

The lesson is that a very high score is not automatically evidence of a good model. If a feature contains information derived from the outcome, the model is effectively given the answer.


In [10]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Example proxy label for the deliberate experiment:
# 1 = observed declining performance, 0 = otherwise.
# This is deliberately constructed from observed outcome information.
feature_frame["label"] = (
    feature_frame["gsc_impressions"]
    < feature_frame["gsc_impressions"].median()
).astype(int)

# DELIBERATE LEAK
feature_frame["leakage_feature"] = feature_frame["label"]

leak_data = feature_frame.dropna(
    subset=["gsc_impressions", "gsc_clicks", "gsc_avg_position",
            "ga4_sessions", "gsc_ctr", "leakage_feature"]
)

X_leak = leak_data[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_sessions",
        "gsc_ctr",
        "leakage_feature",
    ]
]

y_leak = leak_data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X_leak,
    y_leak,
    test_size=0.25,
    random_state=42,
    stratify=y_leak
)

leak_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

leak_model.fit(X_train, y_train)

leak_score = accuracy_score(
    y_test,
    leak_model.predict(X_test)
)

print(f"Score with deliberate leakage: {leak_score:.3f}")

Score with deliberate leakage: 1.000


In [11]:
feature_frame = feature_frame.drop(columns=["leakage_feature"])

print("Leakage feature removed.")
print("Honest feature columns:")
print([
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "gsc_ctr",
])

Leakage feature removed.
Honest feature columns:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'gsc_ctr']


## 4. Data limitation

A key limitation is that the warehouse is an unbalanced panel. Different clients have different depths of search and analytics history, and GA4 availability is not uniform across all rows. Therefore, the March 2026 slice does not necessarily represent every client equally.

The five features provide observed decision-support signals for the available pages, but they cannot by themselves establish that a page will decline in the future. A future-outcome label and a properly separated historical feature window would be required for an honest predictive model.


In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 — Data limits checks

# 1. Check whether the panel is unbalanced:
client_date_coverage = con.execute(f"""
    SELECT
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT report_date) AS dates,
        COUNT(DISTINCT client_hash_id || '_' || CAST(report_date AS VARCHAR)) AS client_date_pairs
    FROM {FACT_DAILY}
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
""").df()

display(client_date_coverage)

# 2. Check the date coverage available in the March slice:
march_window = con.execute(f"""
    SELECT
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT report_date) AS distinct_dates
    FROM {FACT_DAILY}
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
""").df()

display(march_window)

# 3. Check missingness in the metrics used for ranking:
missing_metrics = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_impressions IS NULL) AS missing_impressions,
        COUNT(*) FILTER (WHERE gsc_clicks IS NULL) AS missing_clicks,
        COUNT(*) FILTER (WHERE gsc_avg_position IS NULL) AS missing_avg_position
    FROM {FACT_DAILY}
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
""").df()

display(missing_metrics)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,clients,dates,client_date_pairs
0,55,31,1644


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_date,max_date,distinct_dates
0,2026-03-01,2026-03-31,31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,missing_impressions,missing_clicks,missing_avg_position
0,9841378,0,0,6230317


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.